# 🧠 Train the U-Net segmentation model on BraTS 2020 (GPU)

Run this on **Google Colab** or **Kaggle** with a GPU runtime to train `best_unet.pth`
on real BraTS ground-truth masks. Segmentation is impractical on CPU (24k+ slices),
so a GPU is strongly recommended here.

**Colab:** Runtime → Change runtime type → **GPU (T4)**.
**Kaggle:** Settings → Accelerator → **GPU**.

At the end you can download the trained `best_unet.pth` and drop it into
`backend/weights/` locally so the segmentation runs with real weights.

In [ ]:
# 1) Confirm a GPU is available
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "No GPU — switch the runtime to GPU!")
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# 2) Clone the repository
import os
REPO = "https://github.com/asifkhan78866-cmd/tumb.git"
if not os.path.isdir("tumb"):
    !git clone $REPO
%cd tumb
!git pull --ff-only

In [ ]:
# 3) Install dependencies
!pip install -q -r backend/requirements.txt

In [ ]:
# 4) Kaggle credentials
#    Colab: upload your kaggle.json (Kaggle > Settings > Create New API Token),
#    or paste a new-style token below.
import os

# --- Option A: new-style token (KGAT_...) ---
KAGGLE_API_TOKEN = ""  # <-- paste your KGAT_ token here (optional)
if KAGGLE_API_TOKEN:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
        f.write(KAGGLE_API_TOKEN)
    os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
    os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

# --- Option B: legacy kaggle.json upload (Colab) ---
if not KAGGLE_API_TOKEN:
    try:
        from google.colab import files
        print("Upload your kaggle.json:")
        up = files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        import shutil
        shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    except ImportError:
        print("On Kaggle the API is already authenticated — nothing to do.")

In [ ]:
# 5) Download the BraTS 2020 segmentation dataset (~6.8 GB)
#    On Kaggle you can instead 'Add Data' -> brats2020-training-data and symlink it
#    into backend/dataset/ to skip this download.
!python -m backend.utils.dataset_download --kind segmentation

In [ ]:
# 6) Train the U-Net (GPU + mixed precision, checkpoints best_unet.pth)
#    Tune epochs/batch-size to your GPU. A T4 handles batch-size 32 comfortably.
!python -m backend.training.train_segmentation --epochs 40 --batch-size 32 --workers 2

In [ ]:
# 7) Evaluate (writes Dice + metrics.json, confusion matrix, ROC)
!python -m backend.training.evaluate

In [ ]:
# 8) Preview a few predictions to sanity-check the masks
import glob, torch, numpy as np, matplotlib.pyplot as plt
from backend import config
from backend.models import UNet
from backend.utils.dataset import SegmentationDataset

model = UNet(1, 1, config.BASE_FILTERS).to(config.DEVICE)
model.load_state_dict(torch.load(config.SEG_WEIGHTS_PATH, map_location=config.DEVICE)["model_state"])
model.eval()
ds = SegmentationDataset(config.DATASET_DIR, augment=False)

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for r in range(3):
    x, y = ds[np.random.randint(len(ds))]
    with torch.no_grad():
        pred = torch.sigmoid(model(x.unsqueeze(0).to(config.DEVICE)))[0, 0].cpu().numpy() > 0.5
    axes[r, 0].imshow(x[0], cmap="gray"); axes[r, 0].set_title("MRI (FLAIR)")
    axes[r, 1].imshow(y[0], cmap="gray"); axes[r, 1].set_title("Ground truth")
    axes[r, 2].imshow(pred, cmap="gray"); axes[r, 2].set_title("Prediction")
    for c in range(3):
        axes[r, c].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 9) Download the trained weights, then place them in backend/weights/ locally.
try:
    from google.colab import files
    files.download("backend/weights/best_unet.pth")
except ImportError:
    print("On Kaggle: the file is at backend/weights/best_unet.pth — add it to the")
    print("notebook Output, or copy it to /kaggle/working/ to download from the sidebar.")
    import shutil, os
    if os.path.isdir("/kaggle/working"):
        shutil.copy("backend/weights/best_unet.pth", "/kaggle/working/best_unet.pth")
        print("Copied to /kaggle/working/best_unet.pth")